In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load the raw dataset
# Path is relative to the notebooks folder
try:
    df = pd.read_csv('../data/raw_it_assets.csv')
    print("✅ Data loaded successfully.")
except FileNotFoundError:
    print("❌ File not found. Please check the data folder.")

✅ Data loaded successfully.


In [3]:
# Initial data inspection
df.head()

,Asset_ID,Asset_Type,Assigned_To,Department,Acquisition_Date,Warranty_Expiry_Date,Status,Maintenance_Cost,License_Expiry_Date,Vendor_Name,Location,Compliance_Status,OS_Version,Repair_Count
0,ASSET_0000,Desktop,NaN,Other,2020-05-10,NaN,Decommissioned,NaN,NaN,Cisco,Other,Non-Compliant,Win 10,99
1,ASSET_0001,Laptop,User_187,Finance,2018/11/27,2021-02-27,Stock,180.0,NaN,Other,Other,At Risk,Win 10,3
2,ASSET_0002,Other,User_376,IT,30/08/2024,2026-03-09,In Use,71.0,NaN,Cisco,Cloud,Compliant,Linux-Generic,2
3,ASSET_0003,Monitor,User_799,Other,2023-08-10,2025-11-27,In Use,736.0,NaN,Other,Taichung,Non-Compliant,NaN,3
4,ASSET_0004,Monitor,User_940,Other,2019/10/09,2020-10-28,In Use,650.0,NaN,Lenovo,Hsinchu,At Risk,NaN,0


In [4]:
# Check data types and overall structure
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Asset_ID              500 non-null    object 
 1   Asset_Type            500 non-null    object 
 2   Assigned_To           416 non-null    object 
 3   Department            500 non-null    object 
 4   Acquisition_Date      500 non-null    object 
 5   Warranty_Expiry_Date  450 non-null    object 
 6   Status                500 non-null    object 
 7   Maintenance_Cost      458 non-null    float64
 8   License_Expiry_Date   109 non-null    object 
 9   Vendor_Name           500 non-null    object 
 10  Location              500 non-null    object 
 11  Compliance_Status     500 non-null    object 
 12  OS_Version            287 non-null    object 
 13  Repair_Count          500 non-null    int64  
dtypes: float64(1), int64(1), object(12)
memory usage: 54.8+ KB


### ☞ Data Transformation: Type Casting & Optimization

Before proceeding with data imputation and analysis, it is essential to ensure each column has the correct Dtype. This step is crucial for two reasons:

1. **Arithmetic Readiness:** Converting date-related strings to datetime64 objects allows for time-based calculations (e.g., estimating warranty dates).

2. **Memory Efficiency:** Converting high-cardinality strings (like Department or Asset_Type) into category types significantly reduces memory usage and speeds up visualization rendering.

In [5]:
# Date Conversion
date_cols = ['Acquisition_Date', 'Warranty_Expiry_Date', 'License_Expiry_Date']
for col in date_cols:
    df[col] = pd.to_datetime(df[col].astype(str).str.strip(), errors='coerce', format='mixed')

# 2. Categorical Optimization
cat_cols = ['Asset_Type', 'Department', 'Status', 'Location', 'Vendor_Name', 'Compliance_Status']
for col in cat_cols:
    df[col] = df[col].astype('category')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   Asset_ID              500 non-null    object        
 1   Asset_Type            500 non-null    category      
 2   Assigned_To           416 non-null    object        
 3   Department            500 non-null    category      
 4   Acquisition_Date      500 non-null    datetime64[ns]
 5   Warranty_Expiry_Date  450 non-null    datetime64[ns]
 6   Status                500 non-null    category      
 7   Maintenance_Cost      458 non-null    float64       
 8   License_Expiry_Date   109 non-null    datetime64[ns]
 9   Vendor_Name           500 non-null    category      
 10  Location              500 non-null    category      
 11  Compliance_Status     500 non-null    category      
 12  OS_Version            287 non-null    object        
 13  Repair_Count        

In [6]:
# Check for missing values explicitly
df.isnull().sum()

Asset_ID                  0
Asset_Type                0
Assigned_To              84
Department                0
Acquisition_Date          0
Warranty_Expiry_Date     50
Status                    0
Maintenance_Cost         42
License_Expiry_Date     391
Vendor_Name               0
Location                  0
Compliance_Status         0
OS_Version              213
Repair_Count              0
dtype: int64

### ☞ Data Diagnostics: Insights from Missing Values

After running df.isnull().sum(), the results reveal specific operational patterns within the IT infrastructure rather than random data loss. This highlights the importance of Domain-Specific Imputation over simple data deletion.

1. **Physical Inventory Status (Assigned_To)**

- **Observation:** There are 84 missing entries in the Assigned_To column.

- **Context:** In a real-world IT environment, a missing assignee typically indicates the asset is physically located in the warehouse (buffer stock).

- **Strategy:** I will fill these missing values with "In Stock" to maintain the integrity of the inventory analysis.

2. **Asset Category Logic (License_Expiry_Date & OS_Version)**

- **Observation:** Significant missing values are found in these columns (391 and 213, respectively).

- **Context:** This is logically consistent with the asset types. Peripherals like Monitors do not have Operating Systems, and only Software assets require license tracking.

- **Strategy:** These will be handled using category-aware labels (e.g., "Not Applicable") to prevent bias in the machine learning model.

3. **Maintenance History (Maintenance_Cost)**

- **Observation:** 42 records have missing maintenance costs.

- **Context:** This suggests these assets have not undergone any repairs since acquisition.

- **Strategy:** These will be imputed with 0 to accurately reflect the health and ROI of these "healthy" assets.

4. **Data Recovery: Estimated Warranty (Warranty_Expiry_Date)**

- **Observation:** 50 records are missing warranty expiration dates.

- **Strategy:** I implemented a recovery logic by estimating the expiry as 3 years (1095 days) from the Acquisition Date.

Conclusion: By choosing Logical Imputation instead of dropping rows, I preserve the full 500-record dataset. This allows the predictive model to learn the characteristics of "In-Stock" and "Zero-Maintenance" assets, which are critical for accurate lifecycle forecasting.

In [7]:
# Fill basic categorical and numerical nulls

# Fill Assigned_To: Empty values indicate the asset is physically in the warehouse.
df['Assigned_To'] = df['Assigned_To'].fillna('In-Stock')

# Fill Maintenance_Cost: Empty values suggest no repair costs incurred yet.
df['Maintenance_Cost'] = df['Maintenance_Cost'].fillna(0)

# Fill OS_Version: Non-computing devices (like monitors) don't have an OS.
df['OS_Version'] = df['OS_Version'].fillna('Not Applicable')

In [8]:
# Logic-based imputation for Warranty
# Estimate expiry as 3 years from the Acquisition Date

mask = df['Warranty_Expiry_Date'].isnull()
df.loc[mask, 'Warranty_Expiry_Date'] = df.loc[mask, 'Acquisition_Date'] + pd.Timedelta(days=1095)

df.isnull().sum()

Asset_ID                  0
Asset_Type                0
Assigned_To               0
Department                0
Acquisition_Date          0
Warranty_Expiry_Date      0
Status                    0
Maintenance_Cost          0
License_Expiry_Date     391
Vendor_Name               0
Location                  0
Compliance_Status         0
OS_Version                0
Repair_Count              0
dtype: int64

In [9]:
# Diagnosis: Outliers & Logical Errors
# Repair Count Outliers
df[df['Repair_Count'] > 10][['Asset_ID', 'Repair_Count']].head()

,Asset_ID,Repair_Count
0,ASSET_0000,99
20,ASSET_0020,99
40,ASSET_0040,99
60,ASSET_0060,99
80,ASSET_0080,99


In [10]:
# Retired Assets Still Assigned
retired_conflict = df[(df['Status'].isin(['Retired', 'Decommissioned'])) & 
                      (~df['Assigned_To'].isin(['In-Stock', 'Decommissioned']))]
print(f"Conflict count: {len(retired_conflict)}")
retired_conflict[['Asset_ID', 'Status', 'Assigned_To']].head()

Conflict count: 131


,Asset_ID,Status,Assigned_To
21,ASSET_0021,Decommissioned,User_445
22,ASSET_0022,Retired,User_250
29,ASSET_0029,Decommissioned,User_552
31,ASSET_0031,Retired,User_706
33,ASSET_0033,Decommissioned,User_492


In [11]:
# OS Version on Non-Computing Assets
monitor_os = df[(df['Asset_Type'] == 'Monitor') & (df['OS_Version'] != 'Not Applicable')]
print(f"Monitors with OS: {len(monitor_os)}")

Monitors with OS: 0


Before proceeding to visualization, a final 'Sanity Check' is performed. I address data entry errors (e.g., Repair_Count=99) and enforce business logic, such as ensuring that Retired assets are not linked to active users. This ensures our EDA reflects realistic IT operations.

In [12]:
# Data Rectification

# Fix Repair_Count: Replace 99 with Median
repair_median = df.loc[df['Repair_Count'] < 10, 'Repair_Count'].median()
df.loc[df['Repair_Count'] >= 10, 'Repair_Count'] = repair_median

# Fix Assignment Logic: Retired assets should not have active users
df.loc[df['Status'].isin(['Retired', 'Decommissioned']), 'Assigned_To'] = 'Decommissioned'

# Fix OS Consistency: Ensure Monitors show 'Not Applicable'
df.loc[df['Asset_Type'] == 'Monitor', 'OS_Version'] = 'Not Applicable'

After cleaning and rectifying logical inconsistencies, I proceed to Feature Engineering. In this stage, I derive new metrics from existing raw data to support deeper insights during the EDA phase. Specifically, I will calculate the Asset Age to track hardware longevity and Days to Warranty Expiry to quantify maintenance risks. While these features are essential for future predictive modeling, they currently serve as critical dimensions for identifying cost trends and equipment aging patterns.

In [13]:
# Feature Engineering

# Get current date automatically
current_date = pd.to_datetime('today')

# Calculate Asset Age in years
# Using .dt.days to get integer, then dividing by 365
df['Asset_Age'] = ((current_date - df['Acquisition_Date']).dt.days / 365).round(1)

# Calculate Days to Warranty Expiry
# Positive value means active warranty, negative means expired
df['Days_to_Warranty_Expiry'] = (df['Warranty_Expiry_Date'] - current_date).dt.days

# Create a 'Warranty_Status' flag
# This is a categorical feature derived from the days remaining
df['Warranty_Status'] = df['Days_to_Warranty_Expiry'].apply(lambda x: 'Expired' if x < 0 else 'Active')

df[['Acquisition_Date', 'Warranty_Expiry_Date', 'Asset_Age', 'Days_to_Warranty_Expiry', 'Warranty_Status']].head()

,Acquisition_Date,Warranty_Expiry_Date,Asset_Age,Days_to_Warranty_Expiry,Warranty_Status
0,2020-05-10,2023-05-10,5.9,-1076,Expired
1,2018-11-27,2021-02-27,7.4,-1878,Expired
2,2024-08-30,2026-03-09,1.6,-42,Expired
3,2023-08-10,2025-11-27,2.7,-144,Expired
4,2019-10-09,2020-10-28,6.5,-2000,Expired


In [14]:
# Organize Columns for Better Readability
column_order = [
    'Asset_ID', 'Asset_Type', 'Status', 'Department', 'Assigned_To',
    'Acquisition_Date', 'Asset_Age', 'Warranty_Expiry_Date', 'Days_to_Warranty_Expiry', 'Warranty_Status',
    'Maintenance_Cost', 'Repair_Count', 'Vendor_Name', 'Location', 'Compliance_Status', 'OS_Version'
]

# Apply the new column order
df = df[column_order]

df.head()

,Asset_ID,Asset_Type,Status,Department,Assigned_To,Acquisition_Date,Asset_Age,Warranty_Expiry_Date,Days_to_Warranty_Expiry,Warranty_Status,Maintenance_Cost,Repair_Count,Vendor_Name,Location,Compliance_Status,OS_Version
0,ASSET_0000,Desktop,Decommissioned,Other,Decommissioned,2020-05-10,5.9,2023-05-10,-1076,Expired,0.0,4,Cisco,Other,Non-Compliant,Win 10
1,ASSET_0001,Laptop,Stock,Finance,User_187,2018-11-27,7.4,2021-02-27,-1878,Expired,180.0,3,Other,Other,At Risk,Win 10
2,ASSET_0002,Other,In Use,IT,User_376,2024-08-30,1.6,2026-03-09,-42,Expired,71.0,2,Cisco,Cloud,Compliant,Linux-Generic
3,ASSET_0003,Monitor,In Use,Other,User_799,2023-08-10,2.7,2025-11-27,-144,Expired,736.0,3,Other,Taichung,Non-Compliant,Not Applicable
4,ASSET_0004,Monitor,In Use,Other,User_940,2019-10-09,6.5,2020-10-28,-2000,Expired,650.0,0,Lenovo,Hsinchu,At Risk,Not Applicable


In [15]:
# Check if there are any unexpected nulls
df.isnull().sum()

Asset_ID                   0
Asset_Type                 0
Status                     0
Department                 0
Assigned_To                0
Acquisition_Date           0
Asset_Age                  0
Warranty_Expiry_Date       0
Days_to_Warranty_Expiry    0
Warranty_Status            0
Maintenance_Cost           0
Repair_Count               0
Vendor_Name                0
Location                   0
Compliance_Status          0
OS_Version                 0
dtype: int64

In [16]:
# Preview the finalized data
display(df.head())

,Asset_ID,Asset_Type,Status,Department,Assigned_To,Acquisition_Date,Asset_Age,Warranty_Expiry_Date,Days_to_Warranty_Expiry,Warranty_Status,Maintenance_Cost,Repair_Count,Vendor_Name,Location,Compliance_Status,OS_Version
0,ASSET_0000,Desktop,Decommissioned,Other,Decommissioned,2020-05-10,5.9,2023-05-10,-1076,Expired,0.0,4,Cisco,Other,Non-Compliant,Win 10
1,ASSET_0001,Laptop,Stock,Finance,User_187,2018-11-27,7.4,2021-02-27,-1878,Expired,180.0,3,Other,Other,At Risk,Win 10
2,ASSET_0002,Other,In Use,IT,User_376,2024-08-30,1.6,2026-03-09,-42,Expired,71.0,2,Cisco,Cloud,Compliant,Linux-Generic
3,ASSET_0003,Monitor,In Use,Other,User_799,2023-08-10,2.7,2025-11-27,-144,Expired,736.0,3,Other,Taichung,Non-Compliant,Not Applicable
4,ASSET_0004,Monitor,In Use,Other,User_940,2019-10-09,6.5,2020-10-28,-2000,Expired,650.0,0,Lenovo,Hsinchu,At Risk,Not Applicable


In [17]:
# Strip whitespace from string columns just in case
df['Status'] = df['Status'].str.strip()
df['Department'] = df['Department'].str.strip()

In [18]:
# Display final data shape and stats summary
df.shape

(500, 16)

In [19]:
df.describe()

,Acquisition_Date,Asset_Age,Warranty_Expiry_Date,Days_to_Warranty_Expiry,Maintenance_Cost,Repair_Count
count,500,500.000000,500,500.000000,500.000000,500.000000
mean,2021-07-17 13:23:31.199999744,4.755800,2023-08-23 16:50:52.799999744,-970.298000,719.778000,3.594000
min,2018-01-01 00:00:00,1.400000,2019-04-17 00:00:00,-2560.000000,0.000000,0.000000
25%,2019-11-29 12:00:00,3.000000,2021-11-02 12:00:00,-1629.500000,329.500000,1.750000
50%,2021-08-15 00:00:00,4.700000,2023-09-09 00:00:00,-954.000000,747.000000,4.000000
75%,2023-04-18 06:00:00,6.400000,2025-05-10 18:00:00,-344.250000,1095.750000,6.000000
max,2024-12-01 00:00:00,8.300000,2027-11-07 00:00:00,566.000000,1497.000000,7.000000
std,NaN,1.998543,NaN,756.525369,450.972088,2.285522


In [20]:
# Save the cleaned dataset to the processed folder
df.to_csv('../data/processed_it_assets.csv', index=False)

## ETL Conclusion:

The data preprocessing pipeline is now complete. I have successfully:

1. Cleaned missing values and inconsistent data types.

2. Rectified logical errors in maintenance records and asset status.

3. Engineered time-based features (Asset_Age, Warranty_Status) to support predictive insights.

The resulting dataset is clean, consistent, and ready for advanced exploratory data analysis.